In [0]:
from pyspark.sql.functions import *



In [0]:
SOURCE_TABLE = "workspace.silver.historical_hourly"

TARGET_TABLE = "workspace.gold.weather_condition_summary"

silver_df = spark.table(SOURCE_TABLE)

In [0]:
condition_df = (
    silver_df
    .withColumn(
        "weather_condition",
        when(col("has_snow")==True,"Snow")
        
       
        .when(col("has_precipitation")==True,"Precipitation")
        .otherwise("Clear")
    )
)

In [0]:
weather_summary_df = (
    condition_df
    .groupBy("weather_condition")
    .agg(
        count("*").alias("observations"),

        avg("temperature").alias("avg_temperature"),

        avg("humidity").alias("avg_humidity"),

        avg("wind_speed").alias("avg_wind_speed")
    )
)

In [0]:
weather_summary_df = (
    weather_summary_df
    .withColumn("avg_temperature",round("avg_temperature",2))
    .withColumn("avg_humidity",round("avg_humidity",2))
    .withColumn("avg_wind_speed",round("avg_wind_speed",2))
)

In [0]:
(
    weather_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
print(silver_df.columns)